# Практикум 6. Механизм внимания, позиционное кодирование и контекстные эмбеддинги BERT

В лаборатории 6 вы видели, как местоимение «смотрит» на другие слова предложения, как синусоиды кодируют позицию и как вектор многозначного слова сдвигается к своему значению. Здесь повторим это в коде:

1. посчитаем scaled dot-product attention на `numpy` шаг за шагом: $QK^\top$, деление на $\sqrt{d}$, softmax по строкам, взвешенная сумма $V$;
2. увидим, почему одного слоя внимания мало, чтобы понять, к чему относится местоимение, и как с этим справляются два слоя;
3. разберём многоголовое внимание (размерности тензоров) и маски: причинную (как в GPT) и маску паддинга;
4. построим синусоидальное позиционное кодирование и проверим, что без него внимание не видит порядка слов;
5. получим контекстные векторы многозначного слова: сначала в игрушечной модели, затем (в Colab) в настоящей многоязычной модели XLM-RoBERTa, и сыграем с ней в «угадай слово под маской».

*Практикум к лаборатории 6 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Scaled dot-product attention шаг за шагом

Каждый токен с вектором $x_i$ получает три вектора: **запрос** $q_i = x_i W_Q$ («что я ищу»), **ключ** $k_i = x_i W_K$ («что я могу предложить») и **значение** $v_i = x_i W_V$ («что я передам»). Тогда

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V.$$

Строка $i$ матрицы весов показывает, на какие токены «смотрит» токен $i$; каждая строка в сумме равна 1. Деление на $\sqrt{d}$ не даёт скалярным произведениям расти с размерностью и делать softmax слишком резким.

Возьмём пример из лаборатории: «Кошка не перепрыгнула забор, потому что она устала». У каждого слова 4 признака, заданных вручную: *живое*, *предмет*, *местоимение*, *признак* (слово-сказуемое, которое что-то сообщает о подлежащем). В русском подсказкой служит ещё и род («она»/«он»), но в нашей игрушке рода нет, как нет его в узбекском «u»: модель должна понять по смыслу. Служебные слова и глагол здесь нулевые: для этой задачи они не важны. Матрицы $W$ тоже заданы вручную: местоимение спрашивает «где признак?», признак отвечает, а значение передаёт смысловые признаки *живое* и *предмет*.


Теперь сам механизм. Следим за строкой местоимения «она».

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
FEATS = ["живое", "предмет", "местоим.", "признак"]
LEX = {
    "кошка":   [1, 0, 0, 0],
    "забор":   [0, 1, 0, 0],
    "она":     [0, 0, 1, 0],
    "он":      [0, 0, 1, 0],
    "устала":  [1, 0, 0, 1],   # сказуемое о живом существе
    "высоким": [0, 1, 0, 1],   # сказуемое о предмете
}

def embed(sentence):
    toks = sentence.split()
    X = np.array([LEX.get(t, [0, 0, 0, 0]) for t in toks], dtype=float)   # незнакомые слова — нулевой вектор
    return toks, X

S1 = "кошка не перепрыгнула забор потому что она устала"
S2 = "кошка не перепрыгнула забор потому что он был высоким"
toks, X = embed(S1)
print(pd.DataFrame(X, index=toks, columns=FEATS).astype(int).to_string())

d = 4
W_Q1 = np.zeros((4, d)); W_Q1[2, 0] = 3                    # местоимение спрашивает: «где признак?»
W_K1 = np.zeros((4, d)); W_K1[3, 0] = 3                    # признак отвечает: «я здесь»
W_V1 = np.zeros((4, d)); W_V1[0, 0] = 1; W_V1[1, 1] = 1    # передаём «живое» и «предмет»

def softmax_rows(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)      # вычитаем максимум ради устойчивости
    E = np.exp(Z)
    return E / E.sum(axis=-1, keepdims=True)

Q, K, V = X @ W_Q1, X @ W_K1, X @ W_V1
print("Шаг 1. Q, K, V (n × d):", Q.shape, K.shape, V.shape)
print(pd.DataFrame(np.hstack([Q[:, :1], K[:, :1], V[:, :2]]), index=toks,
                   columns=["q[0]", "k[0]", "v[0]", "v[1]"]).to_string())

scores = Q @ K.T
scaled = scores / np.sqrt(d)
A = softmax_rows(scaled)
out = A @ V

i = toks.index([t for t in toks if t in ("она", "он")][0])
print(f"\nШаг 2–4 для строки «{toks[i]}»:")
print("   q·k      :", scores[i])
print("   / √d     :", scaled[i])
print("   exp      :", np.exp(scaled[i]).round(2), " сумма", np.exp(scaled[i]).sum().round(2))
print("   softmax  :", A[i].round(3))
print("\nШаг 5. Выход = A·V, строка", f"«{toks[i]}»:", ", ".join(f"{f} = {x:.3f}" for f, x in zip(FEATS[:2], out[i, :2])))
print("Суммы строк A:", A.sum(axis=1).round(6))

Местоимение почти всё внимание (0,93) отдало сказуемому «устала» и получило от него признак *живое*. У остальных токенов запрос нулевой, поэтому их внимание равномерное ($1/8$ на каждый токен). Проверим softmax по `scipy` и нарисуем матрицу весов.

In [ ]:
from scipy.special import softmax as sp_softmax

print("softmax совпадает со scipy:", np.allclose(A, sp_softmax(scaled, axis=1)))

def show_attention(ax, A, labels, title):
    ax.imshow(A, cmap="Blues", vmin=0, vmax=1)
    for r in range(A.shape[0]):
        for c in range(A.shape[1]):
            ax.text(c, r, f"{A[r, c]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if A[r, c] > 0.6 else "black")
    ax.set_xticks(range(len(labels)), labels, rotation=60, fontsize=8)
    ax.set_yticks(range(len(labels)), labels, fontsize=8)
    ax.set_xlabel("на кого смотрим (ключи)")
    ax.set_ylabel("кто смотрит (запросы)")
    ax.set_title(title)

fig, ax = plt.subplots(figsize=(6, 5))
show_attention(ax, A, toks, "Веса внимания, один слой")
plt.tight_layout()
plt.show()

### Проверка с PyTorch (только в Colab)

В PyTorch есть готовая функция `torch.nn.functional.scaled_dot_product_attention`: она считает ровно ту же формулу (по умолчанию делит на $\sqrt{d}$, где $d$ — длина запроса). Ячейка ниже сравнивает её с нашим `numpy`.

In [ ]:
try:
    import torch
    import torch.nn.functional as F
    tq, tk, tv = (torch.tensor(M, dtype=torch.float32)[None] for M in (Q, K, V))   # добавили ось пакета: (1, n, d)
    out_torch = F.scaled_dot_product_attention(tq, tk, tv)[0].numpy()
    print("PyTorch:", out_torch.shape, "совпадает с numpy:", np.allclose(out_torch, out, atol=1e-5))
except Exception as e:
    print("PyTorch недоступен:", e)

## 2. Почему одного слоя мало: к чему относится «она»?

Сравним два предложения: «…потому что **она устала**» (речь о кошке) и «…потому что **он был высоким**» (речь о заборе). В одном слое запрос местоимения $q = x W_Q$ зависит только от вектора самого местоимения, а ключи «кошки» и «забора» не зависят от остальных слов. Значит, отношение весов «кошка : забор» в строке местоимения в обоих предложениях одинаковое. Нужен второй слой:

* **слой 1**: местоимение смотрит на сказуемое и получает от него смысловой признак (*живое* у «устала», *предмет* у «высоким»);
* **слой 2**: обогащённое местоимение ищет существительное с этим признаком.

Между слоями работает **остаточная связь** (residual): $h = x + \text{Attention}(x)$, так что исходные признаки токена не теряются. В настоящих трансформерах слоёв 6–24, и такие «цепочки» внимания складываются сами при обучении.

В выводе ниже сравните отношение весов «кошка : забор»: в слое 1 оно равно 1,00 в обоих предложениях, а в слое 2 местоимение уверенно выбирает «кошку» в первом предложении и «забор» во втором.

In [ ]:
def attention(X, W_Q, W_K, W_V, mask=None):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    S = Q @ K.T / np.sqrt(Q.shape[-1])
    if mask is not None:
        S = np.where(mask, S, -1e9)            # запрещённые позиции получают −∞
    A = softmax_rows(S)
    return A @ V, A

# слой 2: запрос — смысловые признаки, ключ — смысловые признаки существительного (не признака и не местоимения)
W_Q2 = np.zeros((4, d)); W_Q2[0, 0] = 3; W_Q2[1, 1] = 3
W_K2 = np.zeros((4, d)); W_K2[0, 0] = 3; W_K2[1, 1] = 3
W_K2[2, :2] = -3; W_K2[3, :2] = -3
W_V2 = np.eye(4)

def two_layers(X):
    out1, A1 = attention(X, W_Q1, W_K1, W_V1)
    H1 = X + out1                               # остаточная связь
    out2, A2 = attention(H1, W_Q2, W_K2, W_V2)
    return H1 + out2, A1, A2

ANIM, OBJ = "кошка", "забор"
results = {}
for s in (S1, S2):
    tk_, X_ = embed(s)
    H2, A1, A2 = two_layers(X_)
    p = [j for j, t in enumerate(tk_) if t in ("она", "он")][0]
    a, o = tk_.index(ANIM), tk_.index(OBJ)
    results[s] = (tk_, A1[p], A2[p])
    print(s)
    print(f"   слой 1: «{tk_[p]}» → " + ", ".join(f"{t} {w:.2f}" for t, w in zip(tk_, A1[p]) if w > 0.05),
          f"|  {ANIM}:{OBJ} = {A1[p, a] / A1[p, o]:.2f}")
    print(f"   слой 2: «{tk_[p]}» → " + ", ".join(f"{t} {w:.2f}" for t, w in zip(tk_, A2[p]) if w > 0.05),
          f"|  {ANIM}:{OBJ} = {A2[p, a] / A2[p, o]:.2f}")

fig, axes = plt.subplots(2, 2, figsize=(11, 5), sharey=True)
for r, s in enumerate((S1, S2)):
    tk_, a1, a2 = results[s]
    for c, (w, name) in enumerate([(a1, "слой 1"), (a2, "слой 2")]):
        ax = axes[r, c]
        cols = ["#D9532C" if t in (ANIM, OBJ) else "#2B55C9" for t in tk_]
        ax.bar(range(len(tk_)), w, color=cols)
        ax.set_xticks(range(len(tk_)), tk_, rotation=30, fontsize=8)
        ax.set_ylim(0, 1)
        pred = [t for t in tk_ if LEX.get(t, [0, 0, 0, 0])[3] == 1][0]      # сказуемое-признак этого предложения
        ax.set_title(f"{name}: …{pred}", fontsize=10)
fig.suptitle("Куда смотрит местоимение")
plt.tight_layout()
plt.show()

## 3. Многоголовое внимание и маски

**Многоголовое внимание** (multi-head): вместо одного внимания размерности $d_{model}$ считают $h$ «голов» размерности $d_k = d_{model}/h$, каждая со своими $W_Q, W_K, W_V$; результаты склеивают и умножают на $W_O$. Разные головы могут следить за разными отношениями (соседнее слово, подлежащее, местоимение…). Проследим размерности тензоров на случайных данных.

In [ ]:
rng = np.random.default_rng(0)
n, d_model, h = 8, 12, 3
d_k = d_model // h
Xr = rng.normal(size=(n, d_model))
W_Q, W_K, W_V, W_O = (rng.normal(size=(d_model, d_model)) / np.sqrt(d_model) for _ in range(4))

Q = Xr @ W_Q
K = Xr @ W_K
V = Xr @ W_V
print("X              ", Xr.shape)
print("Q = X·W_Q      ", Q.shape)
Qh = Q.reshape(n, h, d_k).transpose(1, 0, 2)      # (голова, токен, d_k)
Kh = K.reshape(n, h, d_k).transpose(1, 0, 2)
Vh = V.reshape(n, h, d_k).transpose(1, 0, 2)
print("Q по головам   ", Qh.shape)
S = Qh @ Kh.transpose(0, 2, 1) / np.sqrt(d_k)
print("оценки         ", S.shape)
A_heads = softmax_rows(S)
heads = A_heads @ Vh
print("выход голов    ", heads.shape)
concat = heads.transpose(1, 0, 2).reshape(n, d_model)
print("склейка        ", concat.shape)
Y = concat @ W_O
print("выход = ·W_O   ", Y.shape)

D, H = 768, 12
print(f"\nBERT-base: d_model={D}, h={H}, d_k={D // H}; параметров в одном слое внимания: 4·{D}² + 4·{D} = {4 * D * D + 4 * D:,}")

fig, axes = plt.subplots(1, h, figsize=(11, 3.4))
for k in range(h):
    axes[k].imshow(A_heads[k], cmap="Blues", vmin=0, vmax=1)
    axes[k].set_title(f"голова {k + 1}")
    axes[k].set_xlabel("ключ")
axes[0].set_ylabel("запрос")
plt.tight_layout()
plt.show()

**Маски.** BERT видит предложение целиком (двунаправленное внимание). Генеративная модель (GPT) при предсказании слова $i$ не должна подглядывать в будущие слова, поэтому к оценкам добавляют **причинную маску**: $-\infty$ выше диагонали, после softmax эти веса равны 0. **Маска паддинга** нужна, когда предложения разной длины собраны в один пакет (batch): на токены `[PAD]` смотреть нельзя.

In [ ]:
toks1, X1 = embed(S1)
n1 = len(toks1)
rng = np.random.default_rng(1)
Wq, Wk = rng.normal(size=(4, 4)), rng.normal(size=(4, 4))
X1n = X1 + rng.normal(scale=0.3, size=X1.shape)            # немного шума, чтобы веса не были одинаковыми

causal = np.tril(np.ones((n1, n1), dtype=bool))            # True — смотреть можно
_, A_full = attention(X1n, Wq, Wk, np.eye(4))
_, A_causal = attention(X1n, Wq, Wk, np.eye(4), mask=causal)
print("причинная маска (1 — можно смотреть):")
print(causal.astype(int))
print("строка 3 без маски: ", A_full[2].round(2))
print("строка 3 с маской: ", A_causal[2].round(2))

# маска паддинга: короткое предложение дополнено [PAD] до длины n1
short = "кошка устала".split()
toks2 = short + ["[PAD]"] * (n1 - len(short))
X2 = np.vstack([embed("кошка устала")[1], np.zeros((n1 - len(short), 4))]) + rng.normal(scale=0.3, size=(n1, 4))
pad_mask = np.array([t != "[PAD]" for t in toks2])[None, :].repeat(n1, axis=0)
_, A_pad = attention(X2, Wq, Wk, np.eye(4), mask=pad_mask)
print("\n", toks2)
print("строка 1 с маской паддинга:", A_pad[0].round(2), "сумма", A_pad[0].sum().round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
show_attention(axes[0], A_full, toks1, "Без маски (BERT)")
show_attention(axes[1], A_causal, toks1, "Причинная маска (GPT)")
plt.tight_layout()
plt.show()

## 4. Позиционное кодирование

Внимание — это взвешенная сумма, ей всё равно, в каком порядке стоят токены. Если переставить слова, каждый токен получит тот же выходной вектор (переставленный вместе с ним). Чтобы порядок имел значение, к вектору токена прибавляют вектор позиции. В исходном трансформере он синусоидальный:

$$PE_{(p,\,2i)}=\sin\frac{p}{10000^{2i/d}},\qquad PE_{(p,\,2i+1)}=\cos\frac{p}{10000^{2i/d}}.$$

Младшие координаты колеблются быстро, старшие медленно, как стрелки часов. Скалярное произведение $PE_p\cdot PE_{p+k}=\sum_i\cos(k\,\omega_i)$ зависит только от расстояния $k$, а не от самой позиции $p$. (BERT вместо формулы учит векторы позиций, но идея та же.)

Проверим на двухслойной модели из раздела 2: прочитаем предложение задом наперёд. Затем нарисуем карту позиционных векторов и проверим, что $PE_p\cdot PE_{p+k}$ зависит только от $k$: кривые для $p=5$ и $p=20$ совпадают.

In [ ]:
def positional_encoding(n_pos, d):
    pos = np.arange(n_pos)[:, None]
    i = np.arange(0, d, 2)[None, :]
    angle = pos / 10000 ** (i / d)
    PE = np.zeros((n_pos, d))
    PE[:, 0::2] = np.sin(angle)
    PE[:, 1::2] = np.cos(angle)
    return PE

toks1, X1 = embed(S1)
order = list(range(len(toks1)))[::-1]                       # читаем задом наперёд
p = [j for j, t in enumerate(toks1) if t in ("она", "он")][0]
p_rev = order.index(p)
print("исходный порядок:  ", " ".join(toks1))
print("обратный порядок:", " ".join(toks1[j] for j in order))

H_orig, _, _ = two_layers(X1)
H_rev, _, _ = two_layers(X1[order])
print(f"\nбез позиций: выход «{toks1[p]}» одинаковый?", np.allclose(H_orig[p], H_rev[p_rev]),
      "  max|Δ| =", np.abs(H_orig[p] - H_rev[p_rev]).max().round(4))

PE = positional_encoding(len(toks1), 4)
H_orig_pe, _, _ = two_layers(X1 + PE)
H_rev_pe, _, _ = two_layers(X1[order] + PE)                 # позиции остаются на месте, слова переставлены
print(f"с позициями:   выход «{toks1[p]}» одинаковый?", np.allclose(H_orig_pe[p], H_rev_pe[p_rev]),
      "  max|Δ| =", np.abs(H_orig_pe[p] - H_rev_pe[p_rev]).max().round(4))
print("\nPE для первых 3 позиций (d=4):")
print(PE[:3].round(3))

PE64 = positional_encoding(50, 64)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
im = axes[0].imshow(PE64, cmap="RdBu", aspect="auto", vmin=-1, vmax=1)
axes[0].set_xlabel("координата")
axes[0].set_ylabel("позиция")
axes[0].set_title("Позиционные векторы, d = 64")
plt.colorbar(im, ax=axes[0])
for j in (0, 4, 10, 30):
    axes[1].plot(PE64[:, j], label=f"коорд. {j}")
axes[1].set_xlabel("позиция")
axes[1].legend(fontsize=7)
axes[1].set_title("Быстрые и медленные волны")
k = np.arange(0, 25)
for p0, style in ((5, "-"), (20, "--")):
    axes[2].plot(k, [PE64[p0] @ PE64[p0 + kk] for kk in k], style, label=f"p = {p0}")
axes[2].set_xlabel("расстояние k")
axes[2].set_ylabel("PE_p · PE_{p+k}")
axes[2].legend()
axes[2].set_title("Зависит только от k")
plt.tight_layout()
plt.show()
print("максимальная разница кривых:", np.abs([PE64[5] @ PE64[5 + kk] - PE64[20] @ PE64[20 + kk] for kk in k]).max().round(10))

## 5. Статические и контекстные эмбеддинги

У Word2Vec (практикум 5) у слова «ключ» один вектор на все значения: дверной ключ, родник и «ключ к решению» смешаны в одной точке. В трансформере вектор слова на выходе слоя — смесь векторов слов контекста с весами внимания, поэтому в каждом предложении он свой.

Игрушка: у слов контекста есть статические 2-мерные векторы (заданы вручную, три смысловые группы), «ключ» стоит между группами. Один шаг внимания с $Q=K=V=X$ даёт контекстный вектор $c=\sum_j \alpha_j x_j$.

In [ ]:
import re

static = {
    # дверь
    "квартиры": (0.3, 2.1), "двери": (-0.2, 1.9), "повернул": (0.4, 1.7), "замке": (-0.3, 2.2),
    # родник
    "лесу": (-1.9, -0.8), "бил": (-1.5, -1.2), "холодный": (-1.8, -1.1), "вода": (-1.6, -0.7), "текла": (-1.4, -1.0),
    # решение
    "решению": (1.8, -0.9), "задачи": (1.6, -1.2), "пониманию": (1.5, -0.8), "теоремы": (1.9, -1.1),
}
TARGET = "ключ"
static_target = (0.2, 0.5)       # между группами, ближе к самому частому значению «дверь»
toy_sentences = [
    ("Ключ от квартиры висел у двери", "дверь"),
    ("Он повернул ключ в замке", "дверь"),
    ("В лесу бил холодный ключ", "родник"),
    ("Из ключа текла вода", "родник"),
    ("Ключ к решению задачи нашёлся", "решение"),
    ("Это ключ к пониманию теоремы", "решение"),
]
GROUPS = {"дверь": ["квартиры", "двери", "повернул", "замке"],
          "родник": ["лесу", "бил", "холодный", "вода", "текла"],
          "решение": ["решению", "задачи", "пониманию", "теоремы"]}

def words(s):
    return re.findall(r"\w+", s.lower())

def contextual(sentence):
    ws = [w for w in words(sentence) if w.startswith(TARGET) or w in static]
    Xs = np.array([static_target if w.startswith(TARGET) else static[w] for w in ws])
    t = [j for j, w in enumerate(ws) if w.startswith(TARGET)][0]
    alpha = softmax_rows(Xs[t] @ Xs.T / np.sqrt(2))          # Q = K = V = X
    return alpha @ Xs, ws, alpha

ctx = []
for s, sense in toy_sentences:
    c, ws, alpha = contextual(s)
    ctx.append(c)
    print(f"{s:<36} [{sense}]  α: " + ", ".join(f"{w} {a:.2f}" for w, a in zip(ws, alpha)), f"→ c = {c.round(2)}")
ctx = np.array(ctx)
print("статический вектор «" + TARGET + "» во всех предложениях один:", static_target, "→ косинус между любыми двумя употреблениями = 1.00")

Cn = ctx / np.linalg.norm(ctx, axis=1, keepdims=True)
labels = [f"{i + 1}:{sense}" for i, (_, sense) in enumerate(toy_sentences)]
print("\nкосинусы контекстных векторов:")
print(pd.DataFrame(Cn @ Cn.T, index=labels, columns=labels).round(2).to_string())

fig, ax = plt.subplots(figsize=(6.5, 5.5))
colors = dict(zip(GROUPS, ["#2B55C9", "#17907E", "#D9532C"]))
for g, ws in GROUPS.items():
    pts = np.array([static[w] for w in ws])
    ax.scatter(pts[:, 0], pts[:, 1], color=colors[g], alpha=0.35, label=g)
    for w, (x, y) in zip(ws, pts):
        ax.annotate(w, (x, y), fontsize=7, color="gray", xytext=(3, 3), textcoords="offset points")
ax.scatter(*static_target, marker="*", s=250, color="black", label=TARGET + " (статический)")
for c, (_, sense) in zip(ctx, toy_sentences):
    ax.annotate("", xy=c, xytext=static_target, arrowprops=dict(arrowstyle="->", color=colors[sense]))
    ax.scatter(*c, marker="*", s=120, color=colors[sense])
ax.legend(fontsize=8, loc="lower center")
ax.set_title("Контекстные векторы «" + TARGET + "» сдвигаются к своему значению")
plt.tight_layout()
plt.show()

Статический вектор у всех шести употреблений один (косинус 1,00), а контекстные векторы одного значения близки между собой и далеки от векторов других значений.

### Как это выглядит на настоящей модели (только в Colab)

Возьмём **XLM-RoBERTa base** (`FacebookAI/xlm-roberta-base`): многоязычный BERT-подобный кодировщик, 12 слоёв, около 280 млн параметров, обучен на текстах 100 языков, среди них русский и узбекский. Скачивание около 1,1 ГБ (1–2 минуты в Colab). Если это долго, замените `MODEL_ID` на `distilbert/distilbert-base-multilingual-cased` (около 540 МБ, 6 слоёв). Библиотеки `transformers` и `torch` в Colab уже установлены.

In [ ]:
MODEL_ID = "FacebookAI/xlm-roberta-base"     # ~1,1 ГБ; запасной вариант: "distilbert/distilbert-base-multilingual-cased" (~540 МБ)
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForMaskedLM
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mlm = AutoModelForMaskedLM.from_pretrained(MODEL_ID)
    mlm.eval()
    n_params = sum(p.numel() for p in mlm.parameters())
    print(f"{MODEL_ID}: {mlm.config.num_hidden_layers} слоёв, hidden = {mlm.config.hidden_size}, {n_params / 1e6:.0f} млн параметров, маска = {tok.mask_token}")
    for s, _ in toy_sentences[:2]:
        print("  ", s, "→", tok.tokenize(s))
except Exception as e:
    mlm = None
    print("Не удалось загрузить модель (нужен интернет в Colab):", e)

Модель режет слова на подслова (токены), поэтому вектор слова — среднее векторов его подслов. «Ключ» с заглавной буквы и «ключ», «ключа» могут разрезаться по-разному: посмотрите на список подслов. Возьмём те же шесть предложений и посмотрим на каждом слое: насколько употребления одного значения ближе друг к другу, чем употребления разных значений. Слой 0 — входные эмбеддинги (почти статические: к вектору подслова прибавлен только вектор позиции).

In [ ]:
real_sentences = list(toy_sentences)

def word_vectors(sentence, prefix):
    # векторы слова, начинающегося с prefix, на всех слоях: (слоёв + 1, hidden)
    start = sentence.lower().find(prefix)
    end = start + re.search(r"\w+", sentence[start:]).end()
    enc = tok(sentence, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0].tolist()
    ids = [j for j, (a, b) in enumerate(offsets) if b > a and a < end and b > start]
    with torch.no_grad():
        hs = mlm(**enc, output_hidden_states=True).hidden_states
    return np.stack([h[0, ids].mean(0).numpy() for h in hs]), [tok.convert_ids_to_tokens(int(enc["input_ids"][0, j])) for j in ids]

if globals().get("mlm") is not None:
    try:
        vecs, senses = [], []
        for s, sense in real_sentences:
            v, pieces = word_vectors(s, TARGET.lower())
            vecs.append(v)
            senses.append(sense)
            print(f"{s:<36} подслова: {pieces}")
        vecs = np.array(vecs)                                  # (предложений, слоёв + 1, hidden)
        same = np.array([[a == b for b in senses] for a in senses])
        off = ~np.eye(len(senses), dtype=bool)
        gaps = []
        print("\nслой   своё   чужое   разрыв")
        for L in range(vecs.shape[1]):
            Vn = vecs[:, L] / np.linalg.norm(vecs[:, L], axis=1, keepdims=True)
            C = Vn @ Vn.T
            s_in, s_out = C[same & off].mean(), C[~same].mean()
            gaps.append(s_in - s_out)
            print(f"{L:5d}   {s_in:.3f}  {s_out:.3f}   {s_in - s_out:+.3f}")
        best = int(np.argmax(gaps))
        Vn = vecs[:, best] / np.linalg.norm(vecs[:, best], axis=1, keepdims=True)
        labels = [f"{i + 1}:{s}" for i, s in enumerate(senses)]
        print(f"\nкосинусы на слое {best}:")
        print(pd.DataFrame(Vn @ Vn.T, index=labels, columns=labels).round(2).to_string())
        plt.figure(figsize=(6, 3.2))
        plt.plot(gaps, marker="o")
        plt.xlabel("слой")
        plt.ylabel("своё − чужое (косинус)")
        plt.title("Где различаются значения «" + TARGET + "»")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("Ошибка:", e)

**Угадай слово под маской.** BERT-подобные модели обучали восстанавливать закрытые слова по контексту с обеих сторон (masked language modeling). Посмотрите, какие слова модель считает вероятными. Последние два предложения проверяют то, что мы моделировали в разделе 2: подберёт ли модель «она» или «он».

In [ ]:
real_masked = [
    "Я забыл [MASK] от квартиры дома.",
    "Москва — [MASK] России.",
    "Кошка не перепрыгнула забор, потому что [MASK] устала.",
    "Кошка не перепрыгнула забор, потому что [MASK] был слишком высоким.",
]

if globals().get("mlm") is not None:
    try:
        from transformers import pipeline
        fill = pipeline("fill-mask", model=mlm, tokenizer=tok)
        for s in real_masked:
            s = s.replace("[MASK]", tok.mask_token)
            print(s)
            for pred in fill(s, top_k=5):
                print(f"   {pred['token_str'].strip():<14} {pred['score']:.3f}")
    except Exception as e:
        print("Ошибка:", e)

## 6. Задания

1. **Температура внимания.** В разделе 1 уберите деление на $\sqrt{d}$ и затем разделите оценки на 4 вместо 2. Как меняется вес «устала» в строке местоимения? Посчитайте энтропию этой строки (в битах) в трёх вариантах.
2. **Новое сказуемое.** Добавьте в `LEX` слово «ленивой» с подходящими признаками и прогоните двухслойную модель на предложении «кошка не перепрыгнула забор потому что она была ленивой». Затем на «…потому что он был старым». Верно ли модель находит антецедент?
3. **Головы.** В разделе 3 поставьте `h = 4` и `h = 6`. Выведите размерности `Qh` и `heads`. Почему `h = 5` не работает при `d_model = 12`?
4. **Причинная маска.** Докажите экспериментом, что при причинной маске выход для первых трёх токенов не меняется, если заменить случайными векторами последние пять строк `Xr`. Без маски меняется?
5. **(Colab)** Добавьте ещё по одному предложению на каждое значение многозначного слова в `real_sentences` и найдите слой с наибольшим разрывом «своё − чужое». Совпадает ли он с последним слоем?

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь